# 03. ONNX export, INT8 경량화, 배포 gate

**목표**

- PyTorch checkpoint를 입력·출력 이름이 고정된 ONNX로 내보낸다.
- ONNX checker와 Python↔ONNX 최대 절대 오차로 변환을 검증한다.
- Linear weight를 동적 INT8로 양자화한다.
- 파일 크기와 p50/p95/p99 latency를 비교한다.
- 결과를 `reports/benchmark.json`으로 남긴다.

전체 구현은 `src/export_optimize.py`에 있고 모든 실행문 바로 위에 문법과 작성 이유를 설명했다.

## 1. export와 benchmark를 실행한다

`02_fine_tuning.ipynb`을 먼저 실행해야 한다.

| 인수 | 뜻 |
|---|---|
| `--artifact-dir artifacts` | checkpoint를 읽고 FP32/INT8 ONNX를 저장할 위치 |
| `--report-dir reports` | 측정 JSON을 저장할 위치 |
| `--repeats 500` | warm-up 뒤 각 모델을 반복 실행할 횟수 |

In [ ]:
# %run은 export script를 현재 Jupyter kernel에서 명령행 프로그램처럼 실행한다.
%run src/export_optimize.py --artifact-dir artifacts --report-dir reports --repeats 500

## 2. 산출물과 gate를 읽는다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1~2 | import | JSON parser와 안전한 경로 객체를 가져온다. |
| 3~4 | context manager와 `json.load` | benchmark 보고서를 자동 close하며 읽는다. |
| 5 | dictionary indexing | CI에서 반드시 True여야 하는 gate만 분리해 본다. |
| 6 | `.stat().st_size` | 실제 파일 byte 수를 확인한다. |

In [ ]:
# JSON 보고서를 읽기 위해 Python 표준 json 모듈을 가져온다.
import json
# Path는 파일 존재와 크기를 운영체제 독립적으로 확인한다.
from pathlib import Path
# with 문은 보고서 읽기가 끝나면 file descriptor를 자동으로 닫는다.
with (Path('reports') / 'benchmark.json').open('r', encoding='utf-8') as file:
    # load는 JSON object를 Python dictionary로 변환한다.
    report = json.load(file)
# gates dictionary의 Boolean이 모두 True인지 검토한다.
print('gates=', report['gates'])
# 두 ONNX 파일의 실제 byte 수를 한 줄에서 나란히 출력한다.
print('fp32 bytes=', Path('artifacts/sensor_model.onnx').stat().st_size, 'int8 bytes=', Path('artifacts/sensor_model.int8.onnx').stat().st_size)

## 결과 해석

- `pytorch_vs_onnx_max_abs_error`: FP32 export가 원본 계산을 보존했는지 본다. 이 예제 gate는 `< 1e-4`다.
- `fp32_vs_int8_sample_max_abs_error`: 양자화가 한 검증 입력의 logits를 얼마나 바꿨는지 본다. 제품 승인은 전체 test set 지표를 사용한다.
- `size_ratio_int8_over_fp32`: 1보다 작으면 파일은 줄었지만, 작은 모델은 quantization metadata 때문에 기대만큼 줄지 않을 수 있다.
- `p50/p95/p99`: 작은 모델에서는 runtime 호출 overhead가 계산보다 클 수 있어 INT8이 더 느릴 수도 있다.

노트북을 실행한 PC 결과는 Jetson/Raspberry Pi/Intel NPU 성능을 대신하지 않는다. 동일 artifact를 실제 대상 장치에서 다시 측정한다.

## PTQ에서 QAT로 넘어가는 판단

1. 먼저 FP32 모델 자체가 품질 목표를 만족하는지 확인한다.
2. 대표 calibration 데이터로 정적 PTQ를 시도한다(CNN에 흔한 선택).
3. 클래스별 지표와 worst-case 출력 차이를 조사한다.
4. 민감한 연산을 FP16/FP32로 남기는 mixed precision을 시도한다.
5. 그래도 목표를 못 맞출 때 QAT로 fake quantization을 포함해 재학습한다.
6. QAT export 뒤 대상 runtime이 Q/DQ graph와 연산을 실제 가속하는지 확인한다.

QAT는 정확도를 자동 보장하지 않는다. calibration 분포, backend 연산 지원, scale granularity(per-tensor/per-channel)가 모두 결과에 영향을 준다.

## TensorRT/OpenVINO로 바꾸는 위치

```text
PyTorch checkpoint
  → 검증된 sensor_model.onnx
      ├─ ONNX Runtime CPU/CUDA/TensorRT EP
      ├─ TensorRT engine (Jetson/DRIVE의 실제 환경에서 생성)
      └─ OpenVINO IR/runtime (Intel 대상 장치에서 생성·측정)
```

학습 코드는 그대로 두고 export 뒤의 compiler/runtime adapter만 교체한다. 단, backend가 지원하지 않는 ONNX 연산, dynamic shape, FP16/INT8 정확도는 별도 gate가 필요하다.

## 직접 해 볼 과제

1. batch 크기 1, 8, 32의 latency와 throughput을 따로 측정한다.
2. `intra_op_num_threads`를 1, 2, 4로 바꾸고 p99와 CPU 사용률을 비교한다.
3. test set 전체에서 FP32와 INT8 정확도 및 위험 클래스 recall을 비교한다.
4. `cpp/README.md`를 따라 C++ 출력을 같은 입력의 Python 출력과 비교한다.
5. `ros2/README.md`의 package를 빌드하고 rosbag replay용 전용 message로 확장한다.

**통과 기준**: ONNX checker 성공, Python↔FP32 ONNX 최대 절대 오차 `< 1e-4`, 두 모델 파일과 benchmark JSON 생성. INT8 속도 향상은 필수 통과 조건이 아니라 측정 결과다.